# Clean one dataset

This notebook cleans **one** CSV dataset at a time:

- **Removes missing values** (NaN, empty strings, "N/A", "null", etc.)
- **Drops columns not needed for training** (IDs, payment_mode, location, etc.)
- Keeps only rows where **text** (description/notes) and **category** (label) are present

**Usage:** Set `INPUT_CSV` in the next section to the file you want to clean (path or filename).  
**Output:** `*_cleaned.csv` is written in the same folder as the input.

## 0. Install dependencies (run once)

Run the cell below to install required packages. This notebook needs **pandas**; `re` and `pathlib` are built-in.

In [18]:
# --- This notebook only needs pandas ---
%pip install pandas

# --- For other project notebooks (e.g. random-forest, neural-network) install: ---
# %pip install pandas numpy matplotlib scikit-learn nltk ipykernel

Note: you may need to restart the kernel to use updated packages.


## 1. Choose the dataset to clean

In [4]:
import pandas as pd
import re
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()

# --- Set this to the CSV you want to clean (path or filename) ---
INPUT_CSV = "Personal_Finance_Dataset.csv"   # e.g. "my_data.csv" or "kaggle_download/budget_data.csv"

def resolve_input_path(csv_input):
    """Resolve to a single Path. Accepts filename (searched in project + kaggle_download) or full path."""
    p = Path(csv_input)
    if p.is_absolute() and p.exists():
        return p
    if p.exists():
        return p.resolve()
    # Search in project root and kaggle_download
    for d in [PROJECT_ROOT, PROJECT_ROOT / "kaggle_download"]:
        if not d.exists():
            continue
        candidate = d / p.name
        if candidate.exists():
            return candidate
    return PROJECT_ROOT / p.name  # will raise when read if not found

input_path = resolve_input_path(INPUT_CSV)
print("Input file:", input_path)
if not input_path.exists():
    print("  -> File not found. Change INPUT_CSV and re-run this cell.")

Input file: /home/mukama/Documents/transaction-categorization-main/Personal_Finance_Dataset.csv


## 2. Cleaning configuration

In [5]:
# Columns to DROP (not needed for training)
COLUMNS_TO_DROP = [
    "transaction_id", "user_id", "payment_mode", "location",
    "id", "Id", "index", "Unnamed: 0",
    "Type", "transaction_type",  # optional: keep if you need Income/Expense
]

# Placeholder strings to treat as missing (will become NaN and then drop rows in required cols)
MISSING_VALUES = ["", "N/A", "n/a", "na", "null", "None", "...", "nan", "-"]

# Names that identify the "text" column (description for categorization)
TEXT_COLUMN_NAMES = ["notes", "Transaction Description", "Text", "description", "text", "details", "narration"]

# Names that identify the "category" column (label for training)
CATEGORY_COLUMN_NAMES = ["category", "Category", "Category Id", "categories", "label", "expense type"]

## 3. Cleaning functions

In [6]:
def is_missing(val):
    if pd.isna(val):
        return True
    s = str(val).strip()
    if s in MISSING_VALUES or s.lower() in [m.lower() for m in MISSING_VALUES]:
        return True
    return False


def clean_amount(value):
    """Convert amount to numeric: remove Rs., ₹, $, commas."""
    if pd.isna(value):
        return None
    s = str(value).strip().replace(",", "")
    s = re.sub(r"[Rs\.\s₹$]", "", s, flags=re.IGNORECASE)
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None


def find_text_and_category_columns(df):
    """Return (text_col, category_col)."""
    cols_lower = {c.lower(): c for c in df.columns}
    text_col = None
    for name in TEXT_COLUMN_NAMES:
        if name.lower() in cols_lower:
            text_col = cols_lower[name.lower()]
            break
    if text_col is None and len(df.columns) >= 1:
        text_col = df.columns[0]
    category_col = None
    for name in CATEGORY_COLUMN_NAMES:
        if name.lower() in cols_lower:
            category_col = cols_lower[name.lower()]
            break
    if category_col is None and len(df.columns) >= 2:
        category_col = df.columns[1]
    return text_col, category_col


def clean_one_dataset(df, path, columns_to_drop, missing_values):
    """
    Drop unneeded columns, replace placeholders with NaN, drop rows with NaN in required columns (text + category),
    clean amount if present.
    """
    df = df.copy()
    # Drop columns that exist
    to_drop = [c for c in columns_to_drop if c in df.columns]
    df = df.drop(columns=to_drop, errors="ignore")

    # Replace placeholder missing with NaN
    for col in df.columns:
        df[col] = df[col].apply(lambda x: pd.NA if is_missing(x) else x)

    text_col, category_col = find_text_and_category_columns(df)
    required = []
    if text_col and text_col in df.columns:
        required.append(text_col)
    if category_col and category_col in df.columns:
        required.append(category_col)
    if not required:
        required = list(df.columns[:2]) if len(df.columns) >= 2 else list(df.columns)

    df = df.dropna(subset=required)

    if "amount" in df.columns:
        df["amount"] = df["amount"].apply(clean_amount)
    if "Amount" in df.columns:
        df["Amount"] = df["Amount"].apply(clean_amount)

    return df.reset_index(drop=True)

## 4. Run cleaning on the chosen dataset

In [7]:
path = input_path
results = []
try:
    raw = pd.read_csv(path)
    n_before = len(raw)
    cleaned = clean_one_dataset(raw, path, COLUMNS_TO_DROP, MISSING_VALUES)
    n_after = len(cleaned)
    out_name = path.stem + "_cleaned" + path.suffix
    out_path = path.parent / out_name
    cleaned.to_csv(out_path, index=False)
    results.append({"file": path.name, "before": n_before, "after": n_after, "output": str(out_path)})
    print(f"{path.name}: {n_before} -> {n_after} rows")
    print(f"Saved to: {out_path}")
except Exception as e:
    print(f"Error: {e}")
    results.append({"file": path.name, "error": str(e)})

Personal_Finance_Dataset.csv: 1500 -> 1500 rows
Saved to: /home/mukama/Documents/transaction-categorization-main/Personal_Finance_Dataset_cleaned.csv


## 5. Preview cleaned data

In [36]:
if results and "error" not in results[0]:
    r = results[0]
    print(f"Summary: {r['before']} rows -> {r['after']} rows")
    out_path = Path(r["output"])
    if out_path.exists():
        sample = pd.read_csv(out_path, nrows=10)
        print("Columns:", list(sample.columns))
        display(sample)
else:
    print("Cleaning failed or no file was processed.")

Summary: 2461 rows -> 2461 rows
Columns: ['Date', 'Mode', 'Category', 'Subcategory', 'Note', 'Amount', 'Income/Expense', 'Currency']


,Date,Mode,Category,Subcategory,Note,Amount,Income/Expense,Currency
0,20/09/2018 12:04:08,Cash,Transportation,Train,2 Place 5 to Place 0,300.0,Expense,INR
1,20/09/2018 12:03:15,Cash,Food,snacks,Idli medu Vada mix 2 plates,600.0,Expense,INR
2,19/09/2018,Saving Bank account 1,subscription,Netflix,1 month subscription,1990.0,Expense,INR
3,17/09/2018 23:41:17,Saving Bank account 1,subscription,Mobile Service Provider,Data booster pack,190.0,Expense,INR
4,16/09/2018 17:15:08,Cash,Festivals,Ganesh Pujan,Ganesh idol,2510.0,Expense,INR
5,15/09/2018 06:34:17,Credit Card,subscription,Tata Sky,Permanent Residence - Tata Play recharge,2000.0,Expense,INR
6,14/09/2018 05:39:17,Cash,Transportation,auto,Place 2 station to Permanent Residence,500.0,Expense,INR
7,13/09/2018 21:35:15,Saving Bank account 1,Transportation,Train,2 Place 0 to Place 3,400.0,Expense,INR
8,13/09/2018 21:01:47,Credit Card,Other,NaN,HBR 2 Months subscription,830.0,Expense,INR
9,13/09/2018 21:01:32,Cash,Food,Grocery,1kg atta,460.0,Expense,INR
